**Fig. 3, step 3: reconstruct the 11 stacks in 3D with the DSM and cyto3 model pairs**

Needs a GPU.

In [ ]:
# Working directory: data/fig3/ under the repository root, where the Fig. 3 deposit folder is downloaded
# (decomposed_2D/, bootstrap_*_models/, domain-specialized_*/, benchmark_outputs*/, metrics_outputs/).
# The 11 ground-truth stacks are read from ../ground_truth/zebrafish/{development,held_out}/.
# Set the EPICYCLE_ANALYSES_DATA environment variable to keep the deposit somewhere else.
import os, sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))   # the notebook's own folder, for benchmarking_scripts/
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment.yml").exists() and (p / "data").is_dir()), None)
assert _root is not None, "start Jupyter from inside the EpiCycle-analyses repository"
os.chdir(Path(os.environ.get("EPICYCLE_ANALYSES_DATA", _root / "data")) / "fig3")
print("working directory:", os.getcwd())


# Setups & Imports

In [ ]:
import os
import torch
import multiprocessing as mproc

# IMPORTANT for Jupyter + multiprocessing
mproc.set_start_method("spawn", force=True)

print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("Visible GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))


CUDA_VISIBLE_DEVICES: 0,1,2
Visible GPU count: 3
0 NVIDIA H200
1 NVIDIA H200
2 NVIDIA H200


In [ ]:
import os
from pathlib import Path
import json
import numpy as np
import tifffile
import torch

import segment3D.usegment3d as uSegment3D
import segment3D.parameters as uSegment3D_params

from benchmarking_scripts.multiview_segmentation_utils import segment_one_view_fullZ
from benchmarking_scripts.notebook_workers import gpu_worker_one_view_notebook

# Specify the Paths to Patches

In [ ]:
# the 7 development stacks
patches_seen = [f'../ground_truth/zebrafish/development/{i}' for i in os.listdir('../ground_truth/zebrafish/development') if 'masks' not in i]
patches_seen.sort()

In [ ]:
# the 4 held-out stacks
patches_unseen = [f'../ground_truth/zebrafish/held_out/{i}' for i in os.listdir('../ground_truth/zebrafish/held_out/') if 'mask' not in i]
patches_unseen.sort()

In [ ]:
# all 11 stacks
patches = patches_seen + patches_unseen
patches

['data/patch-1.tif',
 'data/patch-2.tif',
 'data/patch-3.tif',
 'data/patch-4.tif',
 'data/patch-5.tif',
 'data/patch-7.tif',
 'data/patch-8.tif',
 'data_holdout/patch-10.tif',
 'data_holdout/patch-14.tif',
 'data_holdout/patch-6.tif',
 'data_holdout/patch-9.tif']

# Specify the Paths to Model Pairs

In [ ]:
# the round-4 + dense fine-tuned (DSM) pairs
bootstrap_ft_model_paths = 'domain-specialized_finetuned_bootstrap/models/'
bootstrap_axial_ft_models = [f'{bootstrap_ft_model_paths}{i}' for i in os.listdir('domain-specialized_finetuned_bootstrap/models/') 
                             if ('.npy' not in i) and ('axial' in i)]
bootstrap_axial_ft_models.sort()

bootstrap_lateral_ft_models = [f'{bootstrap_ft_model_paths}{i}' for i in os.listdir('domain-specialized_finetuned_bootstrap/models/') 
                             if ('.npy' not in i) and ('lateral' in i)]
bootstrap_lateral_ft_models.sort()

bootstrap_ft_model_pairs = list(zip(bootstrap_lateral_ft_models, bootstrap_axial_ft_models))
bootstrap_ft_model_pairs

[('domain-specialized_finetuned_bootstrap/models/domain-specialized_Bootstrap4Finetuned_lateral_500epoch_scratch_lenient_rescale_True',
  'domain-specialized_finetuned_bootstrap/models/domain-specialized_Bootrap4Finetuned_axial_500epoch_scratch_lenient_rescale_True'),
 ('domain-specialized_finetuned_bootstrap/models/domain-specialized_Bootstrap4Finetuned_lateral_500epoch_scratch_strict_rescale_True',
  'domain-specialized_finetuned_bootstrap/models/domain-specialized_Bootrap4Finetuned_axial_500epoch_scratch_strict_rescale_True')]

In [ ]:
# the Cellpose cyto3 fine-tuned pairs
cyto3_model_paths = 'domain-specialized_finetuned_CPCyto3/models/'
cyto3_axial_ft_models = [f'{cyto3_model_paths}{i}' for i in os.listdir(cyto3_model_paths) if ('.npy' not in i) and ('axial' in i)]
cyto3_axial_ft_models.sort()

cyto3_lateral_ft_models = [f'{cyto3_model_paths}{i}' for i in os.listdir(cyto3_model_paths) if ('.npy' not in i) and ('lateral' in i)]
cyto3_lateral_ft_models.sort()

cyto3_ft_model_pairs = list(zip(cyto3_lateral_ft_models, cyto3_axial_ft_models))
cyto3_ft_model_pairs

[('domain-specialized_finetuned_CPCyto3/models/domain-specialized_lateral_500epoch_scratch_lenient_CPcyto3_Finetuned',
  'domain-specialized_finetuned_CPCyto3/models/domain-specialized_axial_500epoch_scratch_lenient_CPcyto3_Finetuned'),
 ('domain-specialized_finetuned_CPCyto3/models/domain-specialized_lateral_500epoch_scratch_lenient_CPcyto3_Finetuned-HLR',
  'domain-specialized_finetuned_CPCyto3/models/domain-specialized_axial_500epoch_scratch_lenient_CPcyto3_Finetuned-HLR')]

In [ ]:
# final list of models to evaluate
from pathlib import Path

model_pairs = []

# --------------------------------------------------
# 1. Round-4 + dense fine-tuned (DSM) models
# --------------------------------------------------
for i, (lat, ax) in enumerate(bootstrap_ft_model_pairs):
    pair_id = f"bootstrap_models_round4-{i}"
    model_pairs.append({
        "id": pair_id,
        "lateral": lat,
        "axial": ax,
        "rescale": True,
        "group": f"bootstrap_round4_{'lenient' if i % 2 == 0 else 'strict'}",
    })
    

# --------------------------------------------------
# 2. Cellpose cyto3 fine-tuned models
# --------------------------------------------------
for lat, ax in cyto3_ft_model_pairs:
    name = Path(lat).name
    pair_id = name.replace("domain-specialized_lateral_", "")
    model_pairs.append({
        "id": pair_id,
        "lateral": lat,
        "axial": ax,
        "rescale": True,
        "group": f"domain_rescale_true_cyto3ft_lenient",
    })

# --------------------------------------------------
# 3. Sanity check
# --------------------------------------------------
print(f"Total model pairs: {len(model_pairs)}\n")

for mp in model_pairs:
    print(
        f"{mp['id']:<45} | "
        f"group={mp['group']:<20} | "
        f"rescale={mp['rescale']}"
    )


Total model pairs: 4

bootstrap_models_round4-0                     | group=bootstrap_round4_lenient | rescale=True
bootstrap_models_round4-1                     | group=bootstrap_round4_strict | rescale=True
500epoch_scratch_lenient_CPcyto3_Finetuned    | group=domain_rescale_true_cyto3ft_lenient | rescale=True
500epoch_scratch_lenient_CPcyto3_Finetuned-HLR | group=domain_rescale_true_cyto3ft_lenient | rescale=True


# Function to Perform Reconstruction

In [ ]:
def run_one_patch_one_pair_notebook(
    patch_path,
    model_pair,
    out_root,
    base_segment_params,
    reconstruction_params,
    reconstruction_post_params,
    gpu_triplet=(0,1,2),
    verbose=True,
):
    """Reconstruct one stack with one lateral/axial model pair and score nothing yet.

    Runs `gpu_worker_one_view_notebook` for the XY (lateral model), XZ and YZ (axial model)
    views in three processes, one GPU each, collects the per-view probability and flow maps,
    fuses them with u-Segment3D's direct method (`reconstruction_params`), post-processes the
    3D labels (`reconstruction_post_params`), and writes
    `<out_root>/<stack>/<model_id>_mask.tif` (uint32) plus `<model_id>_meta.json`
    (stack, model id and group, rescale flag, the two checkpoint paths, mask path).
    Returns the meta dict.
    """
    import multiprocessing as mp

    segment_params = dict(base_segment_params)
    segment_params["rescale"] = model_pair["rescale"]

    q = mp.Queue()
    procs = []

    jobs = [
        ("xy", model_pair["lateral"], gpu_triplet[0]),
        ("xz", model_pair["axial"],   gpu_triplet[1]),
        ("yz", model_pair["axial"],   gpu_triplet[2]),
    ]

    for view, model_path, gpu in jobs:
        p = mp.Process(
            target=gpu_worker_one_view_notebook,
            kwargs=dict(
                patch_path=patch_path,
                model_path=model_path,
                view=view,
                segment_params=segment_params,
                gpu_id=gpu,
                out_queue=q,
                verbose=verbose,
            ),
        )
        p.start()
        procs.append(p)

    results = {}
    for _ in range(3):
        item = q.get()
        if "error" in item:
            for p in procs:                 # a worker still holding results on the queue
                p.terminate()               # would never exit: stop it before joining
                p.join()
            raise RuntimeError(item)
        results[item["view"]] = item

    for p in procs:
        p.join()
    
    seg3D, (prob3D, grad3D) = uSegment3D.aggregate_2D_to_3D_segmentation_direct_method(
        probs=[
            results["xy"]["probs"],
            results["xz"]["probs"],
            results["yz"]["probs"],
        ],
        gradients=[
            results["xy"]["flows"],
            results["xz"]["flows"],
            results["yz"]["flows"],
        ],
        params=reconstruction_params,
    )
    
    seg3D, _ = uSegment3D.postprocess_3D_cell_segmentation(
        seg3D,
        aggregation_params=reconstruction_params,
        postprocess_params=reconstruction_post_params,
        cell_gradients=grad3D,
        savefolder=None,
        basename=None,
    )

    out_dir = Path(out_root) / Path(patch_path).stem
    out_dir.mkdir(parents=True, exist_ok=True)

    mask_path = out_dir / f"{model_pair['id']}_mask.tif"
    tifffile.imwrite(mask_path, seg3D.astype(np.uint32), bigtiff=True)

    # save the metadata
    meta = {
        "patch": Path(patch_path).stem,
        "patch_path": patch_path,
    
        "model_id": model_pair["id"],
        "model_group": model_pair["group"],
        "rescale": model_pair["rescale"],
    
        "lateral_model": model_pair["lateral"],
        "axial_model": model_pair["axial"],
    
        "mask_path": str(mask_path),
    }
    
    meta_path = out_dir / f"{model_pair['id']}_meta.json"
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)


    return meta


# Set Segmentation & Reconstruction Parameters

In [ ]:
# Keep the parameters here consistent with the other comparisons

# Cellpose prediction parameters (Table S6)
base_segment_params = {
    "channels": [2, 1],
    "normalize": True,
    "cellprob_threshold": 0.0,
    "flow_threshold": 0.4,
    "batch_size": 1024,
}

# define the reconstructions parameters to be used for the fusion method
direct_aggregation_params = uSegment3D_params.get_2D_to_3D_aggregation_params()

# the probability maps are Cellpose cell probabilities
direct_aggregation_params['combine_cell_probs']['cellpose_prob_mask'] = True 
direct_aggregation_params['combine_cell_probs']['ksize'] = 25
direct_aggregation_params['combine_cell_probs']['alpha'] = 0.5
direct_aggregation_params['combine_cell_probs']['smooth_sigma'] = 0.8

direct_aggregation_params['combine_cell_gradients']['ksize'] = 25
direct_aggregation_params['combine_cell_gradients']['smooth_sigma'] = 1.5
direct_aggregation_params['combine_cell_gradients']['post_sigma'] = 0.5

# post process autoinfered binary from probability masks
direct_aggregation_params['postprocess_binary']['binary_fill_holes'] = True
direct_aggregation_params['postprocess_binary']['remove_snall_objects'] = True
direct_aggregation_params['postprocess_binary']['min_size'] = 250

# Gradient descent vector field integration
direct_aggregation_params['gradient_descent']['n_iter'] = 50
direct_aggregation_params['gradient_descent']['momenta'] = 0.98
direct_aggregation_params['gradient_descent']['gradient_decay'] = 0.001

direct_aggregation_params['gradient_descent']['debug_viz'] = False
direct_aggregation_params['gradient_descent']['do_mp'] = True
direct_aggregation_params['gradient_descent']['renorm_gradient'] = False

post_process_params = uSegment3D_params.get_postprocess_segmentation_params()
post_process_params['flow_consistency']['do_flow_remove'] = False  # flow-consistency removal off
post_process_params['flow_consistency']['flow_threshold'] = 1.0
post_process_params['flow_consistency']['n_cpu'] = 24
post_process_params['flow_consistency']['dtform_method'] = 'cellpose_improve'

In [ ]:
# Reconstruct every stack with every model pair
for patch in patches:
    for mpair in model_pairs:
        print(f'running {mpair} on patch:')
        run_one_patch_one_pair_notebook(
            patch_path=patch,
            model_pair=mpair,
            out_root="benchmark_outputs_finetuned",
            base_segment_params=base_segment_params,
            reconstruction_params=direct_aggregation_params,
            reconstruction_post_params=post_process_params,
            verbose=True,
        )


running {'id': 'bootstrap_models_round4-0', 'lateral': 'domain-specialized_finetuned_bootstrap/models/domain-specialized_Bootstrap4Finetuned_lateral_500epoch_scratch_lenient_rescale_True', 'axial': 'domain-specialized_finetuned_bootstrap/models/domain-specialized_Bootrap4Finetuned_axial_500epoch_scratch_lenient_rescale_True', 'rescale': True, 'group': 'bootstrap_round4_lenient'} on patch:


100%|██████████| 556/556 [00:00<00:00, 1373.25it/s]
<env>/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
<env>/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
<env>/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you 

Automatric binary thresholding:  0.7
[374 704 704] [(48, 48), (96, 96), (96, 96)]
using total number of tiles :  64
start dynamics
Tile id 62 0 0
Tile id 61 0 0
Tile id 60 0 0
Tile id 45 0 0
Tile id 44 0 0
Tile id 63 0 0
Tile id 48 0 0
Tile id 49 0 0
Tile id 52 0 0
Tile id 53 0 0
Tile id 56 0 0
Tile id 57 0 0
Tile id 47 6790 6790
Tile id 31 23214 23214
Tile id 15 17982 17982
Tile id 3 8710 8710
Tile id 46 6886 6886
Tile id 7 31444 31444
Tile id 59 47464 47464
Tile id 40 4791 4791
Tile id 41 6028 6028
Tile id 51 60942 60942
Tile id 11 51428 51428
Tile id 58 55748 55748
Tile id 55 107432 107432
Tile id 12 88733 88733
Tile id 14 141264 141264
Tile id 13 140896 140896
Tile id 19 150170 150170
Tile id 36 123167 123167
Tile id 28 197514 197514
Tile id 27 196660 196660
Tile id 50 172548 172548
Tile id 2 148355 148355
Tile id 35 199418 199418
Tile id 32 147387 147387
Tile id 37 171931 171931
Tile id 23 230970 230970
Tile id 33 174011 174011
Tile id 54 215935 215935
Tile id 43 274487 274487
Til

100%|██████████| 556/556 [00:00<00:00, 1388.48it/s]
<env>/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
<env>/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
<env>/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you 

Automatric binary thresholding:  0.7
[374 704 704] [(48, 48), (96, 96), (96, 96)]
using total number of tiles :  64
start dynamics
Tile id 62 0 0
Tile id 61 0 0
Tile id 60 0 0
Tile id 44 0 0
Tile id 45 0 0
Tile id 63 0 0
Tile id 53 0 0
Tile id 49 0 0
Tile id 48 0 0
Tile id 52 0 0
Tile id 56 0 0
Tile id 57 0 0
Tile id 47 5205 5205
Tile id 15 16784 16784
Tile id 3 8550 8550
Tile id 31 22393 22393
Tile id 46 5248 5248
Tile id 7 30376 30376
Tile id 59 52300 52300
Tile id 11 47523 47523
Tile id 41 6116 6116
Tile id 51 62368 62368
Tile id 40 4782 4782
Tile id 58 60932 60932
Tile id 12 89222 89222
Tile id 55 115371 115371
Tile id 13 134935 134935
Tile id 14 134180 134180
Tile id 19 157505 157505
Tile id 27 194847 194847
Tile id 50 174478 174478
Tile id 28 198944 198944
Tile id 36 134717 134717
Tile id 35 207247 207247
Tile id 2 148508 148508
Tile id 32 149739 149739
Tile id 33 166302 166302
Tile id 37 169990 169990
Tile id 23 243478 243478
Tile id 54 226482 226482
Tile id 43 277329 277329
Til

100%|██████████| 556/556 [00:00<00:00, 1393.99it/s]
<env>/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
[XZ] Cellpose:   0%|          | 0/512 [00:00<?, ?it/s]<deps>/cellpose/cellpose/transforms.py:600: UserWarning: 'chan to seg' to seg has value range of ZERO
  warnings.warn("'chan to seg' to seg has value range of ZERO")
<deps>/cellpose/cellpose/transforms.py:602: UserWarning: 'chan2 (opt)' has value range of ZERO, can instead set chan2 to 0
  warnings.warn(
<env>/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using st

... (79 outputs left out)

100%|██████████| 602/602 [00:00<00:00, 757.63it/s]
<env>/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
<env>/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
<env>/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you a

Automatric binary thresholding:  0.6
[397 704 704] [(48, 48), (96, 96), (96, 96)]
using total number of tiles :  64
start dynamics
Tile id 31 0 0
Tile id 15 0 0
Tile id 14 0 0
Tile id 51 0 0
Tile id 13 0 0
Tile id 12 0 0
Tile id 11 0 0
Tile id 7 699 699
Tile id 50 0 0
Tile id 48 0 0
Tile id 49 0 0
Tile id 52 0 0
Tile id 8 0 0
Tile id 9 0 0
Tile id 10 0 0
Tile id 53 676 676
Tile id 6 1223 1223
Tile id 63 9598 9598
Tile id 55 8102 8102
Tile id 47 29227 29227
Tile id 3 23329 23329
Tile id 60 29048 29048
Tile id 28 55559 55559
Tile id 30 69577 69577
Tile id 59 64546 64546
Tile id 54 23854 23854
Tile id 27 81998 81998
Tile id 61 103050 103050
Tile id 62 125427 125427
Tile id 35 124192 124192
Tile id 56 71791 71791
Tile id 29 147424 147424
Tile id 5 91119 91119
Tile id 4 153001 153001
Tile id 44 239232 239232
Tile id 2 160922 160922
Tile id 23 262769 262769
Tile id 19 281659 281659
Tile id 39 293618 293618
Tile id 43 323056 323056
Tile id 46 369002 369002
Tile id 57 330035 330035
Tile id 0 3

100%|██████████| 602/602 [00:00<00:00, 702.11it/s]
<env>/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
<env>/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()
<env>/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you a

Automatric binary thresholding:  0.6
[397 704 704] [(48, 48), (96, 96), (96, 96)]
using total number of tiles :  64
start dynamics
Tile id 15 0 0
Tile id 12 0 0
Tile id 13 0 0
Tile id 14 0 0
Tile id 51 0 0
Tile id 11 0 0
Tile id 31 524 524
Tile id 7 726 726
Tile id 49 0 0
Tile id 48 0 0
Tile id 50 0 0
Tile id 9 0 0
Tile id 10 0 0
Tile id 52 0 0
Tile id 8 0 0
Tile id 53 361 361
Tile id 6 1575 1575
Tile id 55 6914 6914
Tile id 63 10852 10852
Tile id 3 26283 26283
Tile id 47 29136 29136
Tile id 60 25437 25437
Tile id 59 67685 67685
Tile id 28 59177 59177
Tile id 54 19891 19891
Tile id 30 75699 75699
Tile id 27 83216 83216
Tile id 61 97529 97529
Tile id 62 123804 123804
Tile id 56 59550 59550
Tile id 35 135380 135380
Tile id 29 150951 150951
Tile id 5 92990 92990
Tile id 4 155966 155966
Tile id 44 238217 238217
Tile id 2 175178 175178
Tile id 23 273436 273436
Tile id 19 284861 284861
Tile id 39 308035 308035
Tile id 43 317524 317524
Tile id 46 366035 366035
Tile id 57 310059 310059
Tile id